In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import pairwise_distances


In [ ]:

# 1. Load the Data from the image
data = {
    'Height(cm)': [169, 170, 173, 174, 167, 173],
    'Weight(KG)': [58, 55, 57, 56, 51, 64],
    'Class': ['Normal', 'Normal', 'Normal', 'Underweight', 'Underweight', 'Normal']
}
df = pd.DataFrame(data)

# The unknown target point from the bottom row
X_test = pd.DataFrame({'Height(cm)': [170], 'Weight(KG)': [57]})





In [ ]:

# ---------------------------------------------------------
# 2. Pre-Exploratory Data Analysis (EDA)
# ---------------------------------------------------------
print("--- Dataset Overview ---")
print(df.head(6))

print("\n--- Summary Statistics ---")
print(df.describe().round(2))

print("\n--- Class Distribution ---")
print(df['Class'].value_counts())


In [ ]:
# ---------------------------------------------------------
# 3. Manual Distance Verification (Matching the Image)
# ---------------------------------------------------------
# Calculate the Euclidean distance from the test point to all training points
X_train = df[['Height(cm)', 'Weight(KG)']]
y_train = df['Class']

# Calculate Euclidean distances
distances = pairwise_distances(X_test, X_train, metric='euclidean')[0]

# Add distances and rankings to a copy of the dataframe to match the image
df_verification = df.copy()
df_verification['Calculated_Distance'] = np.round(distances, 1)
df_verification['Rank'] = df_verification['Calculated_Distance'].rank(method='min').astype(int)

# Sort by Rank to perfectly mimic the image structure
df_verification = df_verification.sort_values(by='Rank').reset_index(drop=True)

print("\n--- Reconstructed Table with Distances and Ranks ---")
print(df_verification)

In [ ]:

# ---------------------------------------------------------
# 4. Apply KNN Algorithm
# ---------------------------------------------------------
# We will use K=3 (looking at the top 3 nearest neighbors)
k = 3
knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')

# Train the model
knn.fit(X_train, y_train)


In [ ]:

# Make the prediction for Height=170, Weight=57
prediction = knn.predict(X_test)
knn_distances, knn_indices = knn.kneighbors(X_test)

print(f"\n--- KNN Prediction (k={k}) ---")
print(f"Test Point: Height = {X_test.iloc[0,0]} cm, Weight = {X_test.iloc[0,1]} KG")
print(f"The {k} nearest neighbors have classes: {y_train.iloc[knn_indices[0]].values}")
print(f"Predicted Class: **{prediction[0]}**")